In [20]:
import time
import requests
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from io import StringIO
from tqdm import tqdm
from numpy import nan
import pandas as pd

url = requests.get("https://finance.naver.com/sise/sise_market_sum.naver?sosok=0&page=1")
url.text
# html = BeautifulSoup(url.text, 'html.parser')
html = BeautifulSoup(url.text)
html
table = html.find('table', {'class':'type_2'})
table

table_str = str(table)
table_io = StringIO(table_str)

tables = pd.read_html(table_io)[0]
tables = tables[tables['종목명'].notnull()]

# tables = tables.drop(['N', '토론실'], axis=1)
tables = tables.drop(['N', '토론'], axis=1)
tables.head()

kospi_box = []
for page in tqdm(range(1, 50)):
    url = requests.get(f"https://finance.naver.com/sise/sise_market_sum.naver?sosok=0&page={page}")
    html = BeautifulSoup(url.text)
    
    table = html.find('table', {'class':'type_2'})
    table_str = str(table)
    table_io = StringIO(table_str)

    tables = pd.read_html(table_io)[0]
    tables = tables[tables['종목명'].notnull()]
    
    tables = tables.drop(['N', '토론'], axis=1)
    tables['소속'] = 'KOSPI'
    kospi_box.append(tables)
    time.sleep(1)

kosdaq_box = []
for page in tqdm(range(1, 40)):
    url = requests.get(f"https://finance.naver.com/sise/sise_market_sum.naver?sosok=1&page={page}")
    html = BeautifulSoup(url.text)
    
    table = html.find('table', {'class':'type_2'})
    table_str = str(table)
    table_io = StringIO(table_str)

    tables = pd.read_html(table_io)[0]
    tables = tables[tables['종목명'].notnull()]
    
    tables = tables.drop(['N', '토론'], axis=1)
    tables['소속'] = 'KOSDAQ'
    kosdaq_box.append(tables)
    time.sleep(1)

# stock = pd.concat(kospi_box + kosdaq_box, axis=0)
stock = pd.concat(kospi_box + kosdaq_box, ignore_index=True )
stock


100%|██████████| 39/39 [00:51<00:00,  1.32s/it]


,종목명,현재가,전일비,등락률,액면가,시가총액,상장주식수,외국인비율,거래량,PER,ROE,소속
0,삼성전자,149000.0,상승 100,+0.07%,100.0,8820260.0,5919638.0,51.86,11928792.0,30.94,9.03,KOSPI
1,SK하이닉스,765000.0,"상승 9,000",+1.19%,5000.0,5569218.0,728002.0,53.50,1867721.0,15.60,31.06,KOSPI
2,현대차,475000.0,"상승 62,000",+15.01%,5000.0,972599.0,204758.0,35.05,2968496.0,11.94,12.43,KOSPI
3,LG에너지솔루션,387500.0,"하락 3,500",-0.90%,500.0,906750.0,234000.0,4.85,121125.0,-103.55,-4.93,KOSPI
4,삼성전자우,110500.0,하락 700,-0.63%,100.0,901652.0,815975.0,77.77,2026489.0,22.94,NaN,KOSPI
...,...,...,...,...,...,...,...,...,...,...,...,...
4221,더테크놀로지,380.0,보합0,0.00%,500.0,47.0,12418.0,1.43,0.0,-0.31,-47.83,KOSDAQ
4222,인트로메딕,66.0,하락 12,-15.38%,100.0,28.0,42998.0,1.54,4042859.0,-2.06,-101.85,KOSDAQ
4223,대호특수강우,2795.0,하락 25,-0.89%,2500.0,24.0,848.0,0.02,245.0,-5.86,NaN,KOSDAQ
4224,소프트센우,6530.0,하락 240,-3.55%,200.0,23.0,357.0,0.12,612.0,-108.83,NaN,KOSDAQ
